# 04 · Level 1–2 model development on validation data (amendment 1.4)

Sandesh Lamichhane, Assistant Professor, Yenepoya (Deemed to be University).

Trains the five frozen candidates (logistic regression, random forest, XGBoost, MLP, CNN-LSTM) on the CICIDS2017
**train** partition and tunes them on its **validation** partition, in two feature spaces: `cic` (82 shared
CICFlowMeter features, for H1a and H1b) and `unsw_canonical` (mapped features, for H1c). Grids and rules are in
`config/models.yaml`. **No test partition and no external dataset is opened**: they stay sealed until the freeze
(ADR-0007), which comes in the next notebook after I have checked these results.

**Before you start:** *Runtime → Change runtime type →* **T4 GPU** (or any GPU) *and High-RAM*. The MLP and
CNN-LSTM need the GPU; the other models use the CPU cores.

Run every cell **from top to bottom**, one at a time. Nothing to add or edit.

| Step | What it trains | Approximate time |
|---|---|---|
| 1 | connect Drive and GitHub, update the code, run the tests (expected `44 passed` and a GPU name) | 3 min |
| 2–6 | `cic` space: logistic regression, random forest, XGBoost, MLP, CNN-LSTM | 4–7 h in total |
| 7–11 | `unsw_canonical` space: the same five | 2–4 h in total |
| 12 | progress table | 1 min |

**This is long (about 6–11 hours), and you do not have to do it in one sitting.** Every grid entry and every
seed is saved to Drive and GitHub as soon as it finishes. If Colab disconnects or you stop it: open the notebook
again, run step 1, then run the step that was interrupted. It continues from where it stopped, and steps that are
already finished just print `already complete`.

**Where results go:** records in `results/models/development/<space>/<candidate>.json` and the table in
`results/models/development/summary.md` (Drive clone and GitHub); run logs in `results/models/logs/`; trained
model files in `MyDrive/PhD-AI-Cyber/models/development/` (Drive only, their SHA-256 in the records on GitHub).


## Step 1 · Connect and update (expected end: `in sync with GitHub at …`, `44 passed`, and a GPU name)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, pathlib, subprocess
DRIVE_ROOT = '/content/drive/MyDrive/PhD-AI-Cyber'
REPO_DIR   = f'{DRIVE_ROOT}/code/phd-ai-threat-detection'
os.environ['PHD_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['PHD_SCRATCH'] = '/content/scratch'
TOKEN  = userdata.get('GITHUB_TOKEN')
REMOTE = f'https://{TOKEN}@github.com/sandesh20lamichhane/phd-ai-threat-detection.git'

import datetime as _dt, os, pathlib, shutil, subprocess

RECORD_DIRS = ('results/', 'docs/data_cards/', 'config/feature_mapping/')

def git(*args, check=False):
    r = subprocess.run(['git', '-C', REPO_DIR, *args], capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(TOKEN or '\0', '***').strip()
    if check and r.returncode != 0:
        print(out)
        raise SystemExit(f'git {args[0]} failed (see message above)')
    return r.returncode, out

def sync(message=None):
    """Make the Drive copy of the repo identical to GitHub, keeping every result produced here.

    Never merges or rebases, so open notebooks or other local edits cannot block it:
    1. remember result files changed here (results/, docs/data_cards/, config/feature_mapping/)
    2. back up every other local change to MyDrive/PhD-AI-Cyber/backups/sync_<time>/
    3. reset the copy to GitHub's main, put the result files back, commit and push them
    """
    git('config', 'user.name', 'Sandesh Lamichhane')
    git('config', 'user.email', 'san30lamichhane@gmail.com')
    git('rebase', '--abort'); git('merge', '--abort')      # clear any half-finished earlier pull
    for attempt in range(3):
        git('fetch', REMOTE, 'main', check=True)
        base = git('merge-base', 'HEAD', 'FETCH_HEAD')[1] or 'FETCH_HEAD'
        changed = set(git('diff', '--name-only', base)[1].split())
        untracked = set(git('ls-files', '--others', '--exclude-standard')[1].split())
        # untracked non-record files survive the reset unless GitHub has a file at the same path
        untracked = {f for f in untracked if f.startswith(RECORD_DIRS) or git('cat-file', '-e', f'FETCH_HEAD:{f}')[0] == 0}
        changed |= untracked
        keep = sorted(f for f in changed if f.startswith(RECORD_DIRS) and pathlib.Path(REPO_DIR, f).is_file())
        other = sorted(f for f in changed if not f.startswith(RECORD_DIRS) and pathlib.Path(REPO_DIR, f).is_file())
        stamp = _dt.datetime.now(_dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        hold = pathlib.Path('/content/sync_hold') / stamp if pathlib.Path('/content').exists() else pathlib.Path(REPO_DIR).parent / f'.sync_hold_{stamp}'
        for f in keep:
            (hold / f).parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(pathlib.Path(REPO_DIR, f), hold / f)
        if other:
            bak = pathlib.Path(DRIVE_ROOT, 'backups', f'sync_{stamp}')
            for f in other:
                (bak / f).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(pathlib.Path(REPO_DIR, f), bak / f)
            print(f'{len(other)} other local change(s) backed up to {bak} (e.g. notebook outputs)')
        git('reset', '--hard', 'FETCH_HEAD', check=True)
        for f in keep:
            dest = pathlib.Path(REPO_DIR, f)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(hold / f, dest)
        if keep:
            git('add', '--', *keep, check=True)
            if git('diff', '--cached', '--quiet')[0] != 0:
                git('commit', '-m', message or 'Save results produced in Colab', check=True)
        code, out = git('push', REMOTE, 'HEAD:main')
        if code == 0:
            head = git('log', '--oneline', '-1')[1]
            print(f'in sync with GitHub at {head}' + (f'; {len(keep)} result file(s) saved' if keep else ''))
            return
        print('GitHub moved while saving; trying again')
    raise SystemExit('could not save to GitHub after 3 tries: copy the messages above into the project thread')


sync('Save results from an earlier Colab run')      # update the code; results made here are kept
assert pathlib.Path(REPO_DIR, 'src/data/pipeline.py').exists()
%cd $REPO_DIR
!pip -q install -r requirements.txt
!python -m pytest -q tests

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - choose a GPU runtime (see the top of this notebook)')

def push(message):
    """Save results from the Drive copy of the repo to GitHub, so both hold the same records."""
    sync(message)
    print('saved to GitHub:', message)

def run(*args, message):
    """Run one preprocessing command (its output shows below); push the results only if it succeeded."""
    ip = get_ipython()
    ip.system('python scripts/preprocess.py ' + ' '.join(args))
    ok = ip.user_ns.get('_exit_code', 1) == 0
    push(message if ok else message + ' (FAILED, log only)')
    if not ok:
        raise SystemExit('This step failed. Copy the error above into the project thread.')

def develop(*args, message):
    """Train one candidate on CICIDS2017 train + val (resumes if interrupted); push the record if it succeeded."""
    ip = get_ipython()
    ip.system('python scripts/develop.py ' + ' '.join(args))
    ok = ip.user_ns.get('_exit_code', 1) == 0
    push(message if ok else message + ' (FAILED, log only)')
    if not ok:
        raise SystemExit('This step failed. Copy the error above into the project thread.')


## Step 2 · `cic` space (82 shared CICFlowMeter features): logistic regression

In [ ]:
develop('--space', 'cic', '--candidate', 'logreg', message='Develop logreg in cic space on train + val (amendment 1.4)')

## Step 3 · `cic` space (82 shared CICFlowMeter features): random forest

In [ ]:
develop('--space', 'cic', '--candidate', 'random_forest', message='Develop random_forest in cic space on train + val (amendment 1.4)')

## Step 4 · `cic` space (82 shared CICFlowMeter features): XGBoost

In [ ]:
develop('--space', 'cic', '--candidate', 'xgboost', message='Develop xgboost in cic space on train + val (amendment 1.4)')

## Step 5 · `cic` space (82 shared CICFlowMeter features): MLP

In [ ]:
develop('--space', 'cic', '--candidate', 'mlp', message='Develop mlp in cic space on train + val (amendment 1.4)')

## Step 6 · `cic` space (82 shared CICFlowMeter features): CNN-LSTM

In [ ]:
develop('--space', 'cic', '--candidate', 'cnn_lstm', message='Develop cnn_lstm in cic space on train + val (amendment 1.4)')

## Step 7 · `unsw_canonical` space (mapped features): logistic regression

In [ ]:
develop('--space', 'unsw_canonical', '--candidate', 'logreg', message='Develop logreg in unsw_canonical space on train + val (amendment 1.4)')

## Step 8 · `unsw_canonical` space (mapped features): random forest

In [ ]:
develop('--space', 'unsw_canonical', '--candidate', 'random_forest', message='Develop random_forest in unsw_canonical space on train + val (amendment 1.4)')

## Step 9 · `unsw_canonical` space (mapped features): XGBoost

In [ ]:
develop('--space', 'unsw_canonical', '--candidate', 'xgboost', message='Develop xgboost in unsw_canonical space on train + val (amendment 1.4)')

## Step 10 · `unsw_canonical` space (mapped features): MLP

In [ ]:
develop('--space', 'unsw_canonical', '--candidate', 'mlp', message='Develop mlp in unsw_canonical space on train + val (amendment 1.4)')

## Step 11 · `unsw_canonical` space (mapped features): CNN-LSTM

In [ ]:
develop('--space', 'unsw_canonical', '--candidate', 'cnn_lstm', message='Develop cnn_lstm in unsw_canonical space on train + val (amendment 1.4)')

## Step 12 · Progress table

Expected: `complete yes` and `seeds 5` on all ten rows. Then send the finished notebook to the project thread.

In [ ]:
develop('--status', message='Model development summary (validation only)')
from IPython.display import Markdown
Markdown(open('results/models/development/summary.md').read())